# SNI v2 — GAP vs multiresolution

Validation-only seed-42 screening. Model yang dilatih hanya **S2G** dan **S2MR**. Test tetap terkunci; checkpoint disimpan setiap epoch ke Hugging Face.

In [ ]:
# 1/6 - Setup repository, Drive, GPU, dan persistence
from google.colab import drive, userdata
from pathlib import Path
import json, os, subprocess, sys, tarfile, time

drive.mount('/content/drive')
REPO = Path('/content/coffee-bean-classification')
BRANCH = 'agent/sni-instance-crops'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/ediprin/coffee-bean-classification.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

token = userdata.get('HF_TOKEN')
assert token, 'Tambahkan Colab Secret HF_TOKEN dengan akses write.'
os.environ['HF_TOKEN'] = token

DRIVE_DATA = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
V2_ROOT = DRIVE_DATA / 'classification-v2'
IMAGE_ROOT = Path('/content/sni-instance-crops')
OUTPUT_ROOT = Path('/content/drive/MyDrive/sni-v2-multiresolution-v1')
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
assert (V2_ROOT / 'audit.json').is_file(), f'SNI v2 tidak ditemukan: {V2_ROOT}'
print('REPO      :', REPO)
print('MANIFEST  :', V2_ROOT)
print('OUTPUT    :', OUTPUT_ROOT)

In [ ]:
# 2/6 - Pulihkan crop ke SSD Colab (Drive hanya untuk backup)
SHARD_ROOT = DRIVE_DATA / 'shards'
if not (IMAGE_ROOT / 'audit.json').is_file():
    shards = sorted(SHARD_ROOT.glob('crop_shard_*.tar'))
    assert shards, f'Shard tidak ditemukan: {SHARD_ROOT}'
    IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    for index, shard in enumerate(shards, 1):
        with tarfile.open(shard, 'r') as archive:
            archive.extractall(IMAGE_ROOT, filter='data')
        if index % 5 == 0 or index == len(shards):
            print(f'RESTORE {index}/{len(shards)}', flush=True)
    for name in ('audit.json', 'manifest.csv'):
        source = DRIVE_DATA / name
        assert source.is_file(), f'Backup metadata tidak ditemukan: {source}'
        subprocess.run(['cp', str(source), str(IMAGE_ROOT / name)], check=True)
audit = json.loads((IMAGE_ROOT / 'audit.json').read_text())
assert audit['status'] == 'complete' and audit['output_crops'] == 31074
print('DATASET SIAP:', IMAGE_ROOT, '| crops=', audit['output_crops'])

In [ ]:
# 3/6 - Helper progres training
def epoch_status(codes, seeds):
    rows = []
    for code in codes:
        for seed in seeds:
            history = OUTPUT_ROOT / 'outputs' / f'{code}_seed{seed}' / 'history.json'
            if history.is_file():
                try:
                    rows.append(f'{code}-{seed}={len(json.loads(history.read_text()))}/50')
                except (OSError, json.JSONDecodeError):
                    rows.append(f'{code}-{seed}=sync')
    return ', '.join(rows) or 'inisialisasi'

def run_with_progress(command, codes, seeds, label):
    log_path = OUTPUT_ROOT / f'{label.lower()}_console.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('MENJALANKAN:', ' '.join(command), flush=True)
    started = time.monotonic()
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
            text=True,
        )
        while process.poll() is None:
            elapsed = (time.monotonic() - started) / 60
            print(f'[{label} {elapsed:.1f} menit] {epoch_status(codes, seeds)}', flush=True)
            time.sleep(30)
    if process.returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        print('\n'.join(tail))
        raise RuntimeError(f'{label} gagal; lihat {log_path}')
    print(f'{label} SELESAI | {epoch_status(codes, seeds)}', flush=True)

In [ ]:
# 4/6 - Jalankan screening seed 42 (S2G lalu S2MR)
command = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_sni_v2_multiresolution',
    '--image-root', str(IMAGE_ROOT),
    '--manifest-root', str(V2_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--stage', 'screen', '--seeds', '42',
    '--evaluation-split', 'val',
    '--hf-repo', HF_REPO,
    '--hf-namespace', 'sni-classification-v2-multiresolution',
    '--hf-sync-every', '1',
]
run_with_progress(command, ('S2G', 'S2MR'), (42,), 'SCREEN')

In [ ]:
# 5/6 - Tampilkan putusan screening
path = OUTPUT_ROOT / 'val_reports' / 'screen_seed42.json'
report = json.loads(path.read_text())
print('DECISION:', report['decision']['decision'])
print('CRITERIA:', report['decision']['criteria'])
for metric in ('macro_f1', 'hard_class_f1', 'bottom3_class_f1', 'worst_class_f1'):
    row = report['comparison'][metric]
    print(f"{metric:22s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} ({row['delta_mean']:+.2%})")
print('TEST DIBUKA:', report['test_opened'])
print('SAVED:', path)
assert report['decision']['decision'] == 'PASS', 'STOP: multiresolusi gagal screening seed 42.'

In [ ]:
# 6/6 - Konfirmasi independen seed 123 dan 2026; test tetap terkunci
command = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_sni_v2_multiresolution',
    '--image-root', str(IMAGE_ROOT),
    '--manifest-root', str(V2_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--stage', 'confirm', '--seeds', '123', '2026',
    '--evaluation-split', 'val',
    '--hf-repo', HF_REPO,
    '--hf-namespace', 'sni-classification-v2-multiresolution',
    '--hf-sync-every', '1',
]
run_with_progress(command, ('S2G', 'S2MR'), (123, 2026), 'CONFIRM')

path = OUTPUT_ROOT / 'val_reports' / 'confirmation.json'
report = json.loads(path.read_text())
print('\n=== PUTUSAN SEED BARU 123/2026 ===')
print('DECISION:', report['decision']['decision'])
print('CRITERIA:', report['decision']['criteria'])
for metric in ('macro_f1', 'hard_class_f1', 'bottom3_class_f1', 'worst_class_f1'):
    row = report['comparison'][metric]
    print(f"{metric:22s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} ({row['delta_mean']:+.2%})")

print('\n=== AGREGAT DESKRIPTIF 42/123/2026 ===')
for metric in ('macro_f1', 'hard_class_f1', 'bottom3_class_f1', 'worst_class_f1'):
    row = report['all_seed_comparison'][metric]
    print(f"{metric:22s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} ({row['delta_mean']:+.2%} +/- {row['delta_std']:.2%})")
print('TEST DIBUKA:', report['test_opened'])
print('SAVED:', path)